<a href="https://colab.research.google.com/github/bhanu613/alias-aware-technical-skill-extraction/blob/main/notebooks/01%20Data%20Preparation%20and%20Lexicon%20Design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation and Lexicon Design

## Purpose

This notebook creates the reproducible data foundation and frozen lexicon for a
lexicon-based technical-skill extraction study.

It starts from a pinned version of the public Djinni Recruitment Dataset,
filters a defined subset of English data-related IT job postings, and creates
deterministic development and held-out evaluation subsets.

Only the development subset is used for candidate discovery, coverage auditing,
canonical-label selection, and alias decisions. The held-out evaluation subset
is not used to select labels, aliases, matching rules, or lexicon policy.

## Research Question

How much and at what precision cost does alias-aware normalisation improve lexicon-based extraction of canonical
technical skills from English data-related IT job postings?

## Study Design

```text
Pinned source dataset
        ↓
Filtering and role-restricted corpus
        ↓
Development and evaluation subset
        ↓
Lexicon and alias design
        ↓
Frozen 20-label lexicon
        ↓
System A and B Extractors and tests
        ↓
Independent manual gold annotation and final evaluation
```

The 100 held-out evaluation documents are created in this notebook but are not
used in any lexicon or alias decision.

In [1]:
from pathlib import Path

import hashlib
import json
import os
import subprocess
import sys

import pandas as pd


repositoryUrl = (
    "https://github.com/bhanu613/"
    "alias-aware-technical-skill-extraction.git"
)


repositoryFolder = Path(
    "/content/alias-aware-technical-skill-extraction"
)


if not repositoryFolder.exists():

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            repositoryUrl,
            str(repositoryFolder)
        ],
        check=True
    )


os.chdir(
    repositoryFolder
)


requirementsPath = (
    repositoryFolder
    / "requirements.txt"
)


subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirementsPath)
    ],
    check=True
)


dataFolder = repositoryFolder / "data"

configFolder = repositoryFolder / "config"

runtimeFolder = Path(
    "/content/dataPreparationOutputs"
)


runtimeFolder.mkdir(
    parents=True,
    exist_ok=True
)


datasetRepository = (
    "lang-uk/"
    "recruitment-dataset-job-descriptions-english"
)


datasetCommit = (
    "b56a6054c10f1266a141e37c8df66c79ff2863af"
)


datasetParquetUrl = (
    "https://huggingface.co/datasets/"
    f"{datasetRepository}/resolve/"
    f"{datasetCommit}/"
    "data/train-00000-of-00001.parquet"
)


print(
    "Repository folder: "
    f"{repositoryFolder}"
)


print(
    "Pinned dataset repository: "
    f"{datasetRepository}"
)


print(
    "Pinned dataset commit: "
    f"{datasetCommit}"
)

Repository folder: /content/alias-aware-technical-skill-extraction
Pinned dataset repository: lang-uk/recruitment-dataset-job-descriptions-english
Pinned dataset commit: b56a6054c10f1266a141e37c8df66c79ff2863af


## Pinned Source Dataset

The source is loaded from a pinned Hugging Face dataset commit rather than a
moving dataset URL. This ensures that the same source revision is used when
the notebook is rerun.

The pipeline uses deterministic filtering and UUID-hash allocation. No random
sample is used to create the development and held-out evaluation subsets.

In [2]:
rawData = pd.read_parquet(
    datasetParquetUrl
)


requiredRawColumns = {
    "id",
    "Position",
    "Company Name",
    "Primary Keyword",
    "Long Description",
    "Long Description_lang"
}


assert requiredRawColumns.issubset(
    rawData.columns
), (
    "The pinned source dataset is missing one or more "
    "required columns."
)


assert rawData[
    "id"
].notna().all(), (
    "The pinned source dataset contains missing document IDs."
)


print(
    "Pinned raw dataset loaded successfully."
)


print(
    f"Raw rows: {len(rawData)}"
)


print(
    f"Raw columns: {len(rawData.columns)}"
)


display(
    rawData[
        [
            "id",
            "Position",
            "Company Name",
            "Primary Keyword",
            "Long Description_lang"
        ]
    ].head(
        5
    )
)

Pinned raw dataset loaded successfully.
Raw rows: 141897
Raw columns: 10


,id,Position,Company Name,Primary Keyword,Long Description_lang
0,c0ca96e7-85df-50df-a64e-d934cd02a170,10 + Blockchain Nodes / Masternodes to set up,MyCointainer,Sysadmin,en
1,64f4b7ea-36e4-5bdd-a8b1-185f32f7dc7f,10 .NET Developers (Middle and Senior level),TechScout.tech,.NET,en
2,b9a1303e-dd0c-5ed1-8f62-be2bc4c7da4f,"10X Engineer (co-founder, #4 employee, USD 11-...",Innoteka,JavaScript,en
3,99cb3f4a-9b4b-53d9-9a3b-bab2c22da346,16 - Amazon Brand Manager,FirstFive,Marketing,en
4,bc1419f7-28e2-582b-8d53-22e28b2f0210,16 - Amazon Brand Manager,MimirB2B,Marketing,en


## Deterministic Filtering

The study retains English postings in four data-related role families:

- Data Science
- Data Analyst
- Data Engineer
- Python

The filtering procedure removes records with missing identifiers or
descriptions, documents with substantial Cyrillic text, postings outside the
defined description-length range, and duplicate description text.

The resulting corpus is used only as the source for the deterministic
development and evaluation split.

In [3]:
selectedRoleFamilies = [
    "Data Science",
    "Data Analyst",
    "Data Engineer",
    "Python"
]


filteredPool = rawData[
    rawData[
        "Primary Keyword"
    ].isin(
        selectedRoleFamilies
    )
].copy()


roleFilteredCount = len(
    filteredPool
)


filteredPool = filteredPool[
    filteredPool[
        "Long Description_lang"
    ].eq(
        "en"
    )
].copy()


englishFilteredCount = len(
    filteredPool
)


filteredPool = filteredPool.dropna(
    subset=[
        "id",
        "Long Description"
    ]
).copy()


filteredPool[
    "Long Description"
] = filteredPool[
    "Long Description"
].astype(
    str
)


missingValueFilteredCount = len(
    filteredPool
)


filteredPool[
    "charc_len"
] = filteredPool[
    "Long Description"
].str.len()


filteredPool[
    "cyri_ratio"
] = (
    filteredPool[
        "Long Description"
    ].str.count(
        r"[\u0400-\u04FF]"
    )
    / filteredPool[
        "charc_len"
    ]
)


filteredPool = filteredPool[
    filteredPool[
        "cyri_ratio"
    ].le(
        0.005
    )
].copy()


cyrillicFilteredCount = len(
    filteredPool
)


filteredPool = filteredPool[
    filteredPool[
        "charc_len"
    ].between(
        500,
        4000
    )
].copy()


lengthFilteredCount = len(
    filteredPool
)


filteredPool = filteredPool.drop_duplicates(
    subset=[
        "Long Description"
    ]
).copy()


duplicateFilteredCount = len(
    filteredPool
)


filteredPool = filteredPool.sort_values(
    "id"
).reset_index(
    drop=True
)


assert len(filteredPool) == 9222, (
    "Expected 9,222 filtered postings from the pinned "
    "source and frozen filtering policy, but found "
    f"{len(filteredPool)}."
)


filteringSummary = pd.DataFrame(
    {
        "Filtering step": [
            "Selected role families",
            "Declared English descriptions",
            "Non-missing ID and description",
            "At most 0.5% Cyrillic characters",
            "Description length from 500 to 4,000 characters",
            "Exact duplicate descriptions removed"
        ],
        "Documents retained": [
            roleFilteredCount,
            englishFilteredCount,
            missingValueFilteredCount,
            cyrillicFilteredCount,
            lengthFilteredCount,
            duplicateFilteredCount
        ]
    }
)


display(
    filteringSummary
)


roleDistribution = filteredPool[
    "Primary Keyword"
].value_counts().rename_axis(
    "Role family"
).reset_index(
    name="Filtered documents"
)


display(
    roleDistribution
)


print(
    "Final filtered corpus: "
    f"{len(filteredPool)} documents."
)

,Filtering step,Documents retained
0,Selected role families,10215
1,Declared English descriptions,10215
2,Non-missing ID and description,10215
3,At most 0.5% Cyrillic characters,9985
4,"Description length from 500 to 4,000 characters",9222
5,Exact duplicate descriptions removed,9222


,Role family,Filtered documents
0,Python,5170
1,Data Science,2040
2,Data Engineer,1318
3,Data Analyst,694


Final filtered corpus: 9222 documents.


## Development and Held-Out Evaluation Split

Each posting is assigned to a stable hash bucket based on its UUID. The
development subset uses buckets 0–2, while the held-out evaluation subset uses
buckets 50–52.

One posting per employer is retained within each subset. Evaluation employers
are excluded if they appear in the development zone. This reduces employer and
template overlap between lexicon design and final evaluation.

The split is deterministic: rerunning it with the same pinned source data
produces the same document IDs.

In [4]:
def createHashBucket(documentId):

    return int(
        hashlib.md5(
            str(
                documentId
            ).encode()
        ).hexdigest(),
        16
    ) % 100


filteredPool[
    "bucket"
] = filteredPool[
    "id"
].apply(
    createHashBucket
)


filteredPool[
    "company_key"
] = filteredPool[
    "Company Name"
].fillna(
    "__unknown__"
).astype(
    str
).str.strip().str.lower()


developmentZone = filteredPool[
    filteredPool[
        "bucket"
    ].isin(
        [
            0,
            1,
            2
        ]
    )
].sort_values(
    "id"
)


developmentData = developmentZone.drop_duplicates(
    subset=[
        "company_key"
    ],
    keep="first"
).head(
    200
).copy()


evaluationZone = filteredPool[
    filteredPool[
        "bucket"
    ].isin(
        [
            50,
            51,
            52
        ]
    )
].sort_values(
    "id"
)


evaluationZone = evaluationZone[
    ~evaluationZone[
        "company_key"
    ].isin(
        set(
            developmentZone[
                "company_key"
            ]
        )
    )
].copy()


evaluationData = evaluationZone.drop_duplicates(
    subset=[
        "company_key"
    ],
    keep="first"
).head(
    100
).copy()


assert len(developmentData) == 200, (
    "Expected 200 development documents, found "
    f"{len(developmentData)}."
)


assert len(evaluationData) == 100, (
    "Expected 100 held-out evaluation documents, found "
    f"{len(evaluationData)}."
)


assert set(
    developmentData[
        "id"
    ]
).isdisjoint(
    set(
        evaluationData[
            "id"
        ]
    )
), (
    "Document leakage: a document ID occurs in both subsets."
)


assert set(
    developmentData[
        "company_key"
    ]
).isdisjoint(
    set(
        evaluationData[
            "company_key"
        ]
    )
), (
    "Employer leakage: an employer occurs in both subsets."
)


assert evaluationData[
    "company_key"
].is_unique, (
    "More than one evaluation posting belongs to the same employer."
)


assert evaluationData[
    "Long Description"
].is_unique, (
    "Duplicate description text appears in the evaluation subset."
)


evaluationData = evaluationData.sort_values(
    "id"
).reset_index(
    drop=True
)


evaluationData.insert(
    0,
    "Annotation order",
    range(
        1,
        len(evaluationData) + 1
    )
)


splitSummary = pd.DataFrame(
    {
        "Measure": [
            "Filtered corpus",
            "Development documents",
            "Held-out evaluation documents",
            "Shared document IDs",
            "Shared employers"
        ],
        "Value": [
            len(filteredPool),
            len(developmentData),
            len(evaluationData),
            len(
                set(
                    developmentData[
                        "id"
                    ]
                )
                & set(
                    evaluationData[
                        "id"
                    ]
                )
            ),
            len(
                set(
                    developmentData[
                        "company_key"
                    ]
                )
                & set(
                    evaluationData[
                        "company_key"
                    ]
                )
            )
        ]
    }
)


display(
    splitSummary
)


developmentRoleDistribution = developmentData[
    "Primary Keyword"
].value_counts().rename_axis(
    "Role family"
).reset_index(
    name="Development documents"
)


evaluationRoleDistribution = evaluationData[
    "Primary Keyword"
].value_counts().rename_axis(
    "Role family"
).reset_index(
    name="Evaluation documents"
)


display(
    developmentRoleDistribution
)


display(
    evaluationRoleDistribution
)


print(
    "Deterministic development/evaluation split passed."
)

,Measure,Value
0,Filtered corpus,9222
1,Development documents,200
2,Held-out evaluation documents,100
3,Shared document IDs,0
4,Shared employers,0


,Role family,Development documents
0,Python,122
1,Data Science,44
2,Data Engineer,26
3,Data Analyst,8


,Role family,Evaluation documents
0,Python,57
1,Data Science,22
2,Data Engineer,15
3,Data Analyst,6


Deterministic development/evaluation split passed.


## Outputs

The generated files below are written only to the Colab runtime output folder.
They are not written over the committed repository files.

Later sections of this notebook use only `developmentData` for candidate
discovery, coverage auditing, canonical-label selection, and alias decisions.

In [6]:
publicColumns = [
    "id",
    "Position",
    "Company Name",
    "Primary Keyword",
    "charc_len",
    "Long Description"
]


filteredPool[
    publicColumns
    + [
        "bucket"
    ]
].to_csv(
    runtimeFolder
    / "pool9222.csv",
    index=False
)


developmentData[
    publicColumns
].to_csv(
    runtimeFolder
    / "development200.csv",
    index=False
)


evaluationData[
    [
        "Annotation order"
    ]
    + publicColumns
].to_csv(
    runtimeFolder
    / "evaluation100.csv",
    index=False
)


goldAnnotationTemplate = evaluationData[
    [
        "Annotation order",
        "id",
        "Position",
        "charc_len",
        "Long Description"
    ]
].copy()


goldAnnotationTemplate[
    "Gold skills"
] = ""


goldAnnotationTemplate[
    "Uncertain"
] = ""


goldAnnotationTemplate[
    "Annotator note"
] = ""


goldAnnotationTemplate[
    "Annotation status"
] = "not started"


goldAnnotationTemplate.to_csv(
    runtimeFolder
    / "GoldAnnotationTemplate.csv",
    index=False
)


print(
    "Outputs created successfully."
)


print(
    f"Filtered pool: {len(filteredPool)} documents"
)


print(
    f"Development set: {len(developmentData)} documents"
)


print(
    f"Held-out evaluation set: {len(evaluationData)} documents"
)


print(
    "Runtime output folder: "
    f"{runtimeFolder}"
)

Outputs created successfully.
Filtered pool: 9222 documents
Development set: 200 documents
Held-out evaluation set: 100 documents
Runtime output folder: /content/dataPreparationOutputs


## Verification of the Frozen Evaluation Subset

The deterministic split above regenerates the held-out evaluation subset from the
pinned source dataset. This check verifies that the regenerated 100-document
subset exactly matches the committed `evaluation100.csv` artifact later used
for gold annotation and final evaluation.

This verification does not use gold labels, predictions, metrics, or any
held-out performance information.

In [7]:
committedEvaluationPath = (
    dataFolder
    / "evaluation100.csv"
)


assert committedEvaluationPath.exists(), (
    "The committed evaluation100.csv file was not found."
)


committedEvaluationData = pd.read_csv(
    committedEvaluationPath
)


requiredEvaluationColumns = {
    "Annotation order",
    "id",
    "Position",
    "Company Name",
    "Primary Keyword",
    "charc_len",
    "Long Description"
}


assert requiredEvaluationColumns.issubset(
    committedEvaluationData.columns
), (
    "The committed evaluation file is missing one or more "
    "required columns."
)


assert len(committedEvaluationData) == 100, (
    "The committed evaluation file must contain 100 documents."
)


assert evaluationData[
    "Annotation order"
].tolist() == committedEvaluationData[
    "Annotation order"
].tolist(), (
    "Generated and committed evaluation annotation orders differ."
)


assert evaluationData[
    "id"
].tolist() == committedEvaluationData[
    "id"
].tolist(), (
    "Generated and committed evaluation document IDs differ."
)


comparisonColumns = [
    "Position",
    "Company Name",
    "Primary Keyword",
    "charc_len",
    "Long Description"
]


for column in comparisonColumns:

    generatedValues = evaluationData[
        column
    ].fillna(
        ""
    ).astype(
        str
    ).tolist()

    committedValues = committedEvaluationData[
        column
    ].fillna(
        ""
    ).astype(
        str
    ).tolist()

    assert generatedValues == committedValues, (
        "Generated and committed evaluation values differ "
        f"for column: {column}"
    )


print(
    "Generated held-out evaluation subset exactly matches "
    "the committed evaluation100.csv artifact."
)

Generated held-out evaluation subset exactly matches the committed evaluation100.csv artifact.


# Candidate Inventory and Development Coverage

## Development-Only Lexicon Design

This section uses only the 200-document development subset to construct and
audit a bounded technical-skill inventory.

The held-out evaluation subset is not used for candidate selection, alias
selection, matching-policy design, coverage auditing, or lexicon decisions.

The development workflow is:

1. Select a deterministic balanced review sample.
2. Identify candidate technical concepts and observed surface forms through
   structured manual review.
3. Audit candidate forms across all 200 development documents.
4. Select a bounded final canonical inventory.
5. Review possible aliases using development contexts.
6. Freeze the final lexicon before held-out evaluation.

## Structured Balanced Development Review

A deterministic 32-document review sample is selected from the development
subset. Each of the four selected role families contributes eight documents.

This review supports qualitative candidate discovery.

In [8]:
documentsPerRole = 8


structuredReviewSample = (
    developmentData
    .sort_values(
        [
            "Primary Keyword",
            "id"
        ]
    )
    .groupby(
        "Primary Keyword",
        group_keys=False
    )
    .head(
        documentsPerRole
    )
    .copy()
)


assert len(structuredReviewSample) == 32, (
    "Expected 32 structured-review documents, found "
    f"{len(structuredReviewSample)}."
)


reviewRoleCounts = structuredReviewSample[
    "Primary Keyword"
].value_counts()


assert (
    reviewRoleCounts
    == documentsPerRole
).all(), (
    "Each role family must contribute exactly "
    "eight review documents."
)


reviewDistribution = reviewRoleCounts.rename_axis(
    "Role family"
).reset_index(
    name="Review documents"
)


display(
    reviewDistribution
)


reviewSampleDisplay = structuredReviewSample[
    [
        "id",
        "Position",
        "Company Name",
        "Primary Keyword",
        "charc_len"
    ]
].sort_values(
    [
        "Primary Keyword",
        "id"
    ]
).reset_index(
    drop=True
)


display(
    reviewSampleDisplay
)


structuredReviewSample.to_csv(
    runtimeFolder
    / "StructuredDevelopmentReview32.csv",
    index=False
)


print(
    "Structured development review created: "
    f"{len(structuredReviewSample)} documents."
)

,Role family,Review documents
0,Data Analyst,8
1,Data Engineer,8
2,Data Science,8
3,Python,8


,id,Position,Company Name,Primary Keyword,charc_len
0,15138580-11ef-58b8-a322-4635cb307455,Dispute Operations Analyst,Sift,Data Analyst,1617
1,40ee8585-cbbb-5ce0-85fd-eea66c440884,Data Analyst,DocuSketch,Data Analyst,692
2,56fee1fc-662b-566b-9cf8-367446a1bce9,Middle - Senior Informatica MDM Developer,Synergetica,Data Analyst,1779
3,82789a78-bb42-501b-9661-980627c6a782,Data Analyst,Sierentz Global Merchants,Data Analyst,1465
4,ab2bf30e-b379-5c26-92a5-27e8d457d51d,Senior Project Manager (MT) with clinical expe...,promdex,Data Analyst,1791
5,ad85aad8-bee0-5fdf-b342-bf8d5e0b0d9e,Senior Web and Data Analyst,ARENA CS,Data Analyst,1881
6,bbbb8b10-ecd7-5ad7-9016-670d3873e1c2,Junior Data Annotator (French or Spanish),Data Science UA,Data Analyst,1951
7,d0ea3869-1112-5f86-8b1f-2bb2b3aeed4c,"Data Analyst in Budapest, Hungary",TempoEast,Data Analyst,3576
8,1b6392b4-ba3d-5bab-9ebd-43841f361b80,"Data Engineer, Music Streaming Technologies (P...",DataArt,Data Engineer,3858
9,1b820c07-63fa-53da-9b46-dfc4f8983c94,Senior Big Data Engineer,Binariks,Data Engineer,2640


Structured development review created: 32 documents.


## Candidate Discovery Record

The following candidate inventory records the outcome of structured manual
review of the balanced development sample.

Candidate discovery is qualitative. The list is not claimed to be an automatic
or complete discovery of every possible technical skill in the corpus.

The review identified 39 provisional technical concepts and 50 observed or
possible surface forms. A surface form may be the canonical wording itself or
a possible alternate expression. An alias is not a separate skill label.

In [14]:
candidateRows = [
    (
        "python",
        "Programming language",
        "python"
    ),
    (
        "sql",
        "Query language",
        "sql"
    ),
    (
        "r",
        "Programming language",
        "r"
    ),
    (
        "javascript",
        "Programming language",
        "javascript"
    ),
    (
        "java",
        "Programming language",
        "java"
    ),
    (
        "c++",
        "Programming language",
        "c++"
    ),
    (
        "golang",
        "Programming language",
        "golang"
    ),
    (
        "typescript",
        "Programming language",
        "typescript"
    ),
    (
        "scala",
        "Programming language",
        "scala"
    ),
    (
        "amazon web services",
        "Cloud platform",
        "aws; amazon web services"
    ),
    (
        "azure",
        "Cloud platform",
        "azure"
    ),
    (
        "google cloud platform",
        "Cloud platform",
        "gcp; google cloud platform"
    ),
    (
        "postgresql",
        "Relational database",
        "postgresql; postgres"
    ),
    (
        "mysql",
        "Relational database",
        "mysql"
    ),
    (
        "mongodb",
        "NoSQL database",
        "mongodb; mongo"
    ),
    (
        "redis",
        "In-memory database",
        "redis"
    ),
    (
        "elasticsearch",
        "Search and analytics engine",
        "elasticsearch"
    ),
    (
        "apache spark",
        "Data-processing framework",
        "spark; pyspark"
    ),
    (
        "apache kafka",
        "Event-streaming platform",
        "kafka"
    ),
    (
        "apache airflow",
        "Workflow orchestration tool",
        "airflow"
    ),
    (
        "hadoop",
        "Big-data framework",
        "hadoop"
    ),
    (
        "bigquery",
        "Cloud data warehouse",
        "bigquery"
    ),
    (
        "databricks",
        "Data and AI platform",
        "databricks"
    ),
    (
        "docker",
        "Container platform",
        "docker"
    ),
    (
        "kubernetes",
        "Container orchestration platform",
        "kubernetes; k8s"
    ),
    (
        "terraform",
        "Infrastructure-as-code tool",
        "terraform"
    ),
    (
        "git",
        "Version-control system",
        "git"
    ),
    (
        "linux",
        "Operating system",
        "linux"
    ),
    (
        "power bi",
        "Business intelligence tool",
        "power bi; microsoft power bi"
    ),
    (
        "tableau",
        "Business intelligence tool",
        "tableau"
    ),
    (
        "jupyter",
        "Computing environment",
        "jupyter; jupyter notebook"
    ),
    (
        "scikit-learn",
        "Machine-learning library",
        "scikit-learn; sklearn"
    ),
    (
        "tensorflow",
        "Machine-learning framework",
        "tensorflow"
    ),
    (
        "pytorch",
        "Machine-learning framework",
        "pytorch"
    ),
    (
        "pandas",
        "Data-analysis library",
        "pandas"
    ),
    (
        "numpy",
        "Numerical-computing library",
        "numpy"
    ),
    (
        "machine learning",
        "Technical concept",
        "machine learning; ml"
    ),
    (
        "natural language processing",
        "Technical concept",
        "natural language processing; nlp"
    ),
    (
        "etl",
        "Data-engineering concept",
        "etl"
    )
]


candidatePool = pd.DataFrame(
    candidateRows,
    columns=[
        "Provisional canonical label",
        "Category",
        "Observed or possible forms"
    ]
)


candidatePool[
    "Discovery source"
] = (
    "Structured 32-document development review"
)


candidatePool[
    "Initial review status"
] = "Coverage audit pending"


candidateFormCount = sum(
    len(
        [
            form.strip()
            for form in forms.split(
                ";"
            )
            if form.strip()
        ]
    )
    for forms in candidatePool[
        "Observed or possible forms"
    ]
)


assert len(candidatePool) == 39, (
    "Expected 39 provisional candidate concepts."
)


assert candidateFormCount == 50, (
    "Expected 50 observed or possible surface forms."
)


candidateCategorySummary = candidatePool[
    "Category"
].value_counts().rename_axis(
    "Category"
).reset_index(
    name="Candidate concepts"
)


display(
    candidateCategorySummary
)


display(
    candidatePool
)


candidatePool.to_csv(
    runtimeFolder
    / "CandidatePoolV1.csv",
    index=False
)


print(
    "Candidate concepts: "
    f"{len(candidatePool)}"
)


print(
    "Observed or possible surface forms: "
    f"{candidateFormCount}"
)

,Category,Candidate concepts
0,Programming language,8
1,Cloud platform,3
2,Relational database,2
3,Business intelligence tool,2
4,Technical concept,2
5,Machine-learning framework,2
6,Search and analytics engine,1
7,Query language,1
8,NoSQL database,1
9,In-memory database,1


,Provisional canonical label,Category,Observed or possible forms,Discovery source,Initial review status
0,python,Programming language,python,Structured 32-document development review,Coverage audit pending
1,sql,Query language,sql,Structured 32-document development review,Coverage audit pending
2,r,Programming language,r,Structured 32-document development review,Coverage audit pending
3,javascript,Programming language,javascript,Structured 32-document development review,Coverage audit pending
4,java,Programming language,java,Structured 32-document development review,Coverage audit pending
5,c++,Programming language,c++,Structured 32-document development review,Coverage audit pending
6,golang,Programming language,golang,Structured 32-document development review,Coverage audit pending
7,typescript,Programming language,typescript,Structured 32-document development review,Coverage audit pending
8,scala,Programming language,scala,Structured 32-document development review,Coverage audit pending
9,amazon web services,Cloud platform,aws; amazon web services,Structured 32-document development review,Coverage audit pending


Candidate concepts: 39
Observed or possible surface forms: 50


## Development Coverage Audit

Every candidate surface form is audited across all 200 development documents.

The audit records document-level coverage and short real-document contexts. At this
point, alternate forms remain candidate forms rather than accepted aliases.

The purpose is to provide evidence for later manual decisions about:

- The bounded 20-label canonical inventory.
- Which alternate forms should be accepted as System B aliases.
- Which forms should be rejected because they are unsupported, redundant, or
  outside the frozen matching policy.

The held-out evaluation subset is not used in this audit.

In [15]:
import re
import unicodedata


def normaliseDevelopmentText(text):

    normalisedText = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    normalisedText = normalisedText.lower()

    normalisedText = normalisedText.replace(
        "\u2019",
        "'"
    ).replace(
        "\u2018",
        "'"
    )

    normalisedText = re.sub(
        r"[\u2013\u2014]",
        "-",
        normalisedText
    )

    normalisedText = re.sub(
        r"\s+",
        " ",
        normalisedText
    )

    return normalisedText.strip()


def createCandidatePattern(candidateForm):

    escapedForm = re.escape(
        candidateForm.lower().strip()
    )

    escapedForm = escapedForm.replace(
        r"\ ",
        r"\s+"
    )

    leftBoundary = r"(?<![A-Za-z0-9+#&_.\-])"

    rightBoundary = r"(?![A-Za-z0-9+#&_\\-])"

    return re.compile(
        leftBoundary
        + escapedForm
        + rightBoundary
    )


normalisedDevelopmentData = developmentData.copy()


normalisedDevelopmentData[
    "normalised_description"
] = normalisedDevelopmentData[
    "Long Description"
].fillna(
    ""
).apply(
    normaliseDevelopmentText
)


assert len(normalisedDevelopmentData) == 200, (
    "Expected 200 normalised development documents."
)


print(
    "Development descriptions prepared for coverage audit: "
    f"{len(normalisedDevelopmentData)}"
)

Development descriptions prepared for coverage audit: 200


In [16]:
coverageRows = []


for _, candidate in candidatePool.iterrows():

    canonicalLabel = candidate[
        "Provisional canonical label"
    ]

    category = candidate[
        "Category"
    ]

    candidateForms = [
        form.strip().lower()
        for form in candidate[
            "Observed or possible forms"
        ].split(
            ";"
        )
        if form.strip()
    ]

    for candidateForm in candidateForms:

        candidatePattern = createCandidatePattern(
            candidateForm
        )

        matchedDocumentIds = []

        exampleContexts = []

        for _, document in normalisedDevelopmentData.iterrows():

            documentText = document[
                "normalised_description"
            ]

            match = candidatePattern.search(
                documentText
            )

            if match:

                matchedDocumentIds.append(
                    document[
                        "id"
                    ]
                )

                if len(exampleContexts) < 2:

                    contextStart = max(
                        0,
                        match.start() - 70
                    )

                    contextEnd = min(
                        len(documentText),
                        match.end() + 120
                    )

                    exampleContexts.append(
                        documentText[
                            contextStart:contextEnd
                        ]
                    )

        coverageRows.append(
            {
                "Provisional canonical label": canonicalLabel,
                "Category": category,
                "Candidate surface form": candidateForm,
                "Form type": (
                    "Canonical candidate"
                    if candidateForm == canonicalLabel
                    else "Alternate candidate form"
                ),
                "Development document count": len(
                    matchedDocumentIds
                ),
                "Example context 1": (
                    exampleContexts[0]
                    if len(exampleContexts) >= 1
                    else ""
                ),
                "Example context 2": (
                    exampleContexts[1]
                    if len(exampleContexts) >= 2
                    else ""
                ),
                "Decision status": "To be reviewed"
            }
        )


coverageAudit = pd.DataFrame(
    coverageRows
)


coverageAudit = coverageAudit.sort_values(
    [
        "Development document count",
        "Provisional canonical label",
        "Candidate surface form"
    ],
    ascending=[
        False,
        True,
        True
    ]
).reset_index(
    drop=True
)


assert len(coverageAudit) == 50, (
    "Expected one coverage-audit row for each of the "
    "50 candidate surface forms."
)


assert coverageAudit[
    "Candidate surface form"
].is_unique, (
    "Each candidate surface form should occur once "
    "in the coverage audit."
)


coverageAudit.to_csv(
    runtimeFolder
    / "CandidateCoverageAudit.csv",
    index=False
)


coverageSummary = pd.DataFrame(
    {
        "Measure": [
            "Development documents audited",
            "Candidate concepts",
            "Candidate surface forms",
            "Forms with at least one development match",
            "Forms with zero development matches"
        ],
        "Value": [
            len(normalisedDevelopmentData),
            len(candidatePool),
            len(coverageAudit),
            int(
                coverageAudit[
                    "Development document count"
                ].gt(
                    0
                ).sum()
            ),
            int(
                coverageAudit[
                    "Development document count"
                ].eq(
                    0
                ).sum()
            )
        ]
    }
)


display(
    coverageSummary
)

,Measure,Value
0,Development documents audited,200
1,Candidate concepts,39
2,Candidate surface forms,50
3,Forms with at least one development match,50
4,Forms with zero development matches,0


In [17]:
mostFrequentCandidateForms = coverageAudit[
    [
        "Provisional canonical label",
        "Candidate surface form",
        "Form type",
        "Development document count"
    ]
].head(
    30
).copy()


zeroCoverageForms = coverageAudit.loc[
    coverageAudit[
        "Development document count"
    ].eq(
        0
    ),
    [
        "Provisional canonical label",
        "Candidate surface form",
        "Form type"
    ]
].copy()


print(
    "Most frequent candidate forms in the "
    "200-document development subset:"
)


display(
    mostFrequentCandidateForms
)


print(
    "Candidate forms with zero development coverage:"
)


display(
    zeroCoverageForms
)


print(
    "Candidate surface forms audited: "
    f"{len(coverageAudit)}"
)

Most frequent candidate forms in the 200-document development subset:


,Provisional canonical label,Candidate surface form,Form type,Development document count
0,python,python,Canonical candidate,172
1,sql,sql,Canonical candidate,82
2,amazon web services,aws,Alternate candidate form,72
3,docker,docker,Canonical candidate,50
4,machine learning,machine learning,Canonical candidate,43
5,kubernetes,kubernetes,Canonical candidate,34
6,postgresql,postgresql,Canonical candidate,34
7,linux,linux,Canonical candidate,29
8,machine learning,ml,Alternate candidate form,28
9,apache spark,spark,Alternate candidate form,27


Candidate forms with zero development coverage:


,Provisional canonical label,Candidate surface form,Form type


Candidate surface forms audited: 50


## Canonical Inventory Decision

The structured review and full development-coverage audit produced 39
provisional technical concepts. The final experiment uses a deliberately bounded
inventory of 20 canonical labels.

A candidate was retained when it was relevant to the selected role families,
clearly defined as a technical concept, feasible to annotate consistently, and
useful for comparing strict canonical matching with alias-aware matching.

Excluding a candidate does not imply that the technology is unimportant. It
means that it was outside the bounded final inventory for this experiment.

In [18]:
finalCanonicalInventory = [
    "python",
    "sql",
    "amazon web services",
    "google cloud platform",
    "azure",
    "postgresql",
    "mongodb",
    "elasticsearch",
    "apache spark",
    "apache kafka",
    "apache airflow",
    "docker",
    "kubernetes",
    "terraform",
    "git",
    "pandas",
    "power bi",
    "scikit-learn",
    "machine learning",
    "natural language processing"
]


assert len(finalCanonicalInventory) == 20, (
    "The final canonical inventory must contain exactly 20 labels."
)


assert len(
    set(
        finalCanonicalInventory
    )
) == 20, (
    "The final canonical inventory must not contain duplicate labels."
)


assert set(
    finalCanonicalInventory
).issubset(
    set(
        candidatePool[
            "Provisional canonical label"
        ]
    )
), (
    "Every final canonical label must originate from "
    "the development candidate pool."
)


candidateDecisions = candidatePool.copy()


candidateDecisions[
    "Final decision"
] = candidateDecisions[
    "Provisional canonical label"
].apply(
    lambda label: (
        "Retained in final 20-label inventory"
        if label in finalCanonicalInventory
        else "Excluded from bounded final inventory"
    )
)


candidateDecisions[
    "Decision rationale"
] = candidateDecisions[
    "Provisional canonical label"
].apply(
    lambda label: (
        "Relevant to the selected role families and retained "
        "for canonicalisation, annotation, and alias-aware "
        "extraction comparison."
        if label in finalCanonicalInventory
        else
        "Not selected for the bounded final experiment. "
        "This decision does not imply that the technology "
        "is unimportant or invalid."
    )
)


retainedCandidates = candidateDecisions.loc[
    candidateDecisions[
        "Final decision"
    ].eq(
        "Retained in final 20-label inventory"
    )
].copy()


excludedCandidates = candidateDecisions.loc[
    candidateDecisions[
        "Final decision"
    ].eq(
        "Excluded from bounded final inventory"
    )
].copy()


assert len(retainedCandidates) == 20, (
    "Expected 20 retained canonical labels."
)


assert len(excludedCandidates) == 19, (
    "Expected 19 excluded candidate concepts."
)


candidateDecisions.to_csv(
    runtimeFolder
    / "CandidateInventoryDecisions.csv",
    index=False
)


candidateDecisionSummary = pd.DataFrame(
    {
        "Decision": [
            "Provisional candidate concepts",
            "Retained canonical labels",
            "Excluded candidate concepts"
        ],
        "Count": [
            len(candidateDecisions),
            len(retainedCandidates),
            len(excludedCandidates)
        ]
    }
)


display(
    candidateDecisionSummary
)


display(
    retainedCandidates[
        [
            "Provisional canonical label",
            "Category",
            "Observed or possible forms",
            "Decision rationale"
        ]
    ].sort_values(
        "Provisional canonical label"
    ).reset_index(
        drop=True
    )
)


print(
    "Final canonical inventory frozen at "
    f"{len(finalCanonicalInventory)} labels."
)

,Decision,Count
0,Provisional candidate concepts,39
1,Retained canonical labels,20
2,Excluded candidate concepts,19


,Provisional canonical label,Category,Observed or possible forms,Decision rationale
0,amazon web services,Cloud platform,aws; amazon web services,Relevant to the selected role families and ret...
1,apache airflow,Workflow orchestration tool,airflow,Relevant to the selected role families and ret...
2,apache kafka,Event-streaming platform,kafka,Relevant to the selected role families and ret...
3,apache spark,Data-processing framework,spark; pyspark,Relevant to the selected role families and ret...
4,azure,Cloud platform,azure,Relevant to the selected role families and ret...
5,docker,Container platform,docker,Relevant to the selected role families and ret...
6,elasticsearch,Search and analytics engine,elasticsearch,Relevant to the selected role families and ret...
7,git,Version-control system,git,Relevant to the selected role families and ret...
8,google cloud platform,Cloud platform,gcp; google cloud platform,Relevant to the selected role families and ret...
9,kubernetes,Container orchestration platform,kubernetes; k8s,Relevant to the selected role families and ret...


Final canonical inventory frozen at 20 labels.


## Focused Power BI Alternate-Form Audit

The full candidate coverage audit includes the canonical phrase `power bi` and
the longer form `microsoft power bi`. This focused development-only check
examines three additional possible forms:

- `powerbi`
- `power-bi`
- `pbi`

These remain candidate forms until their development evidence and contextual
meaning have been reviewed.

In [19]:
powerBiCandidateForms = [
    "powerbi",
    "power-bi",
    "pbi"
]


powerBiRows = []


for candidateForm in powerBiCandidateForms:

    candidatePattern = createCandidatePattern(
        candidateForm
    )

    matchedDocumentIds = []

    exampleContexts = []

    for _, document in normalisedDevelopmentData.iterrows():

        documentText = document[
            "normalised_description"
        ]

        match = candidatePattern.search(
            documentText
        )

        if match:

            matchedDocumentIds.append(
                document[
                    "id"
                ]
            )

            if len(exampleContexts) < 3:

                contextStart = max(
                    0,
                    match.start() - 80
                )

                contextEnd = min(
                    len(documentText),
                    match.end() + 140
                )

                exampleContexts.append(
                    documentText[
                        contextStart:contextEnd
                    ]
                )

    powerBiRows.append(
        {
            "Candidate form": candidateForm,
            "Development document count": len(
                matchedDocumentIds
            ),
            "Example contexts": "\n---\n".join(
                exampleContexts
            ),
            "Review status": "Pending final alias decision"
        }
    )


powerBiAlternateAudit = pd.DataFrame(
    powerBiRows
)


assert powerBiAlternateAudit.loc[
    powerBiAlternateAudit[
        "Candidate form"
    ].eq(
        "powerbi"
    ),
    "Development document count"
].iloc[
    0
] == 1, (
    "Expected powerbi to occur in one development document."
)


assert powerBiAlternateAudit.loc[
    powerBiAlternateAudit[
        "Candidate form"
    ].eq(
        "power-bi"
    ),
    "Development document count"
].iloc[
    0
] == 0, (
    "Expected power-bi to have zero development-document matches."
)


assert powerBiAlternateAudit.loc[
    powerBiAlternateAudit[
        "Candidate form"
    ].eq(
        "pbi"
    ),
    "Development document count"
].iloc[
    0
] == 0, (
    "Expected pbi to have zero development-document matches."
)


powerBiAlternateAudit.to_csv(
    runtimeFolder
    / "PowerBiAlternateFormAudit.csv",
    index=False
)


display(
    powerBiAlternateAudit[
        [
            "Candidate form",
            "Development document count",
            "Review status"
        ]
    ]
)


print(
    "Focused Power BI alternate-form audit completed."
)

,Candidate form,Development document count,Review status
0,powerbi,1,Pending final alias decision
1,power-bi,0,Pending final alias decision
2,pbi,0,Pending final alias decision


Focused Power BI alternate-form audit completed.


## Development Context Review for Candidate Aliases

Candidate-form frequency is evidence, but it is not an alias-acceptance rule.

For each candidate alias, the notebook retrieves up to five deterministic
development-document contexts. The reviewed contexts are used to establish
whether the form refers to the intended canonical technical concept and whether
it adds value beyond strict canonical matching.

A candidate alias is accepted only when:

1. Its reviewed contexts support the intended technical meaning.
2. It represents the same concept as the canonical label.
3. It adds extraction value beyond System A canonical-only matching.
4. It is not redundant with an already matched canonical phrase.
5. It can be represented under the frozen normalisation and boundary policy.

In [20]:
candidateAliases = {
    "amazon web services": [
        "aws"
    ],
    "google cloud platform": [
        "gcp"
    ],
    "postgresql": [
        "postgres"
    ],
    "mongodb": [
        "mongo"
    ],
    "apache spark": [
        "spark",
        "pyspark"
    ],
    "apache kafka": [
        "kafka"
    ],
    "apache airflow": [
        "airflow"
    ],
    "kubernetes": [
        "k8s"
    ],
    "power bi": [
        "powerbi"
    ],
    "scikit-learn": [
        "sklearn"
    ],
    "machine learning": [
        "ml"
    ],
    "natural language processing": [
        "nlp"
    ]
}


aliasContextRows = []


for canonicalLabel, aliases in candidateAliases.items():

    for alias in aliases:

        aliasPattern = createCandidatePattern(
            alias
        )

        contextCount = 0

        for _, document in normalisedDevelopmentData.iterrows():

            documentText = document[
                "normalised_description"
            ]

            match = aliasPattern.search(
                documentText
            )

            if match:

                contextStart = max(
                    0,
                    match.start() - 100
                )

                contextEnd = min(
                    len(documentText),
                    match.end() + 180
                )

                aliasContextRows.append(
                    {
                        "Canonical label": canonicalLabel,
                        "Candidate alias": alias,
                        "Document id": document[
                            "id"
                        ],
                        "Position": document[
                            "Position"
                        ],
                        "Role family": document[
                            "Primary Keyword"
                        ],
                        "Context": documentText[
                            contextStart:contextEnd
                        ]
                    }
                )

                contextCount += 1

                if contextCount == 5:
                    break


aliasContextReview = pd.DataFrame(
    aliasContextRows
)


aliasContextReview = aliasContextReview.sort_values(
    [
        "Canonical label",
        "Candidate alias",
        "Document id"
    ]
).reset_index(
    drop=True
)


assert set(
    aliasContextReview[
        "Candidate alias"
    ]
) == {
    "aws",
    "gcp",
    "postgres",
    "mongo",
    "spark",
    "pyspark",
    "kafka",
    "airflow",
    "k8s",
    "powerbi",
    "sklearn",
    "ml",
    "nlp"
}, (
    "The context-review output must contain all 13 "
    "candidate aliases."
)


aliasContextCounts = aliasContextReview.groupby(
    [
        "Canonical label",
        "Candidate alias"
    ]
).size().reset_index(
    name="Development contexts retrieved"
)


display(
    aliasContextCounts
)


aliasContextExampleDisplay = aliasContextReview.groupby(
    [
        "Canonical label",
        "Candidate alias"
    ],
    as_index=False
).first()[
    [
        "Canonical label",
        "Candidate alias",
        "Position",
        "Role family",
        "Context"
    ]
]


display(
    aliasContextExampleDisplay
)


aliasContextReview.to_csv(
    runtimeFolder
    / "AliasContextReview.csv",
    index=False
)


print(
    "Candidate aliases with retrieved development contexts: "
    f"{len(aliasContextCounts)}"
)

,Canonical label,Candidate alias,Development contexts retrieved
0,amazon web services,aws,5
1,apache airflow,airflow,5
2,apache kafka,kafka,5
3,apache spark,pyspark,5
4,apache spark,spark,5
5,google cloud platform,gcp,5
6,kubernetes,k8s,3
7,machine learning,ml,5
8,mongodb,mongo,4
9,natural language processing,nlp,5


,Canonical label,Candidate alias,Position,Role family,Context
0,amazon web services,aws,Senior Data Science,Data Science,sing data • good communication skills nice to ...
1,apache airflow,airflow,Data engineer,Data Science,"cassandra, etc.)- experience with scraping sys..."
2,apache kafka,kafka,Senior Python Developer for Guardicore,Python,ce working with one or more of the following t...
3,apache spark,pyspark,Senior Full-Stack Engineer (Python/React),Python,tional databases. proficient in sql experience...
4,apache spark,spark,Junior/Middle Data Science Engineer,Data Science,"and speak clearly, easily communicating compl..."
5,google cloud platform,gcp,Back- End Developer,Python,"of specific languages such as elixir, golang,..."
6,kubernetes,k8s,Python Developer (Data-driven),Python,approach experience in aws cloud computing inf...
7,machine learning,ml,Data Scientist,Data Science,te users. what you'll do: - collaborate with d...
8,mongodb,mongo,Junior Data Analyst,Python,trong python knowledge (aggregation frameworks...
9,natural language processing,nlp,Senior ML Engineer (GPT LNN NLP AI),Python,etc. we are seeking a highly skilled and exper...


Candidate aliases with retrieved development contexts: 13


### Recorded Context-Review Conclusions

The table above contains retrieved development-document contexts. The following
review notes record the conclusion of manual contextual review.

These notes do not create new aliases. They document why each candidate alias
was interpreted as referring to its intended canonical concept before the final
accept/reject decision.

In [21]:
aliasReviewNotes = {
    "aws": (
        "Reviewed contexts use AWS to refer to Amazon Web Services "
        "cloud services or infrastructure."
    ),
    "gcp": (
        "Reviewed contexts use GCP to refer to Google Cloud Platform."
    ),
    "postgres": (
        "Reviewed contexts use Postgres to refer to PostgreSQL databases."
    ),
    "mongo": (
        "Reviewed contexts use Mongo to refer to MongoDB or Mongo DB."
    ),
    "spark": (
        "Reviewed contexts use Spark to refer to Apache Spark "
        "data-processing technology."
    ),
    "pyspark": (
        "Reviewed contexts use PySpark to refer to the Python interface "
        "for Apache Spark."
    ),
    "kafka": (
        "Reviewed contexts use Kafka to refer to Apache Kafka "
        "messaging or streaming technology."
    ),
    "airflow": (
        "Reviewed contexts use Airflow to refer to Apache Airflow "
        "workflow orchestration."
    ),
    "k8s": (
        "Reviewed contexts use K8s to refer to Kubernetes "
        "container orchestration."
    ),
    "powerbi": (
        "Reviewed context uses PowerBI to refer to the Power BI "
        "business-intelligence tool."
    ),
    "sklearn": (
        "Reviewed context uses sklearn to refer to the scikit-learn "
        "machine-learning library."
    ),
    "ml": (
        "Reviewed contexts use ML to refer to machine learning."
    ),
    "nlp": (
        "Reviewed contexts use NLP to refer to natural language processing."
    )
}


assert set(
    aliasContextReview[
        "Candidate alias"
    ]
) == set(
    aliasReviewNotes
), (
    "Every context-reviewed candidate alias must have "
    "a recorded review conclusion."
)


aliasContextReview[
    "Review decision"
] = (
    "Confirmed: intended technical meaning"
)


aliasContextReview[
    "Review note"
] = aliasContextReview[
    "Candidate alias"
].map(
    aliasReviewNotes
)


assert aliasContextReview[
    "Review note"
].notna().all(), (
    "Every retrieved context must receive a review note."
)


assert aliasContextReview[
    "Review decision"
].eq(
    "Confirmed: intended technical meaning"
).all(), (
    "Every retrieved context must record the completed "
    "technical-meaning review conclusion."
)


aliasReviewDisplay = aliasContextReview[
    [
        "Canonical label",
        "Candidate alias",
        "Position",
        "Review decision",
        "Review note"
    ]
].drop_duplicates(
    subset=[
        "Canonical label",
        "Candidate alias"
    ]
).sort_values(
    [
        "Canonical label",
        "Candidate alias"
    ]
).reset_index(
    drop=True
)


display(
    aliasReviewDisplay
)


aliasContextReview.to_csv(
    runtimeFolder
    / "CompletedAliasContextReview.csv",
    index=False
)


print(
    "Completed alias context review for "
    f"{len(aliasReviewNotes)} candidate aliases."
)

,Canonical label,Candidate alias,Position,Review decision,Review note
0,amazon web services,aws,Senior Data Science,Confirmed: intended technical meaning,Reviewed contexts use AWS to refer to Amazon W...
1,apache airflow,airflow,Data engineer,Confirmed: intended technical meaning,Reviewed contexts use Airflow to refer to Apac...
2,apache kafka,kafka,Senior Python Developer for Guardicore,Confirmed: intended technical meaning,Reviewed contexts use Kafka to refer to Apache...
3,apache spark,pyspark,Senior Full-Stack Engineer (Python/React),Confirmed: intended technical meaning,Reviewed contexts use PySpark to refer to the ...
4,apache spark,spark,Junior/Middle Data Science Engineer,Confirmed: intended technical meaning,Reviewed contexts use Spark to refer to Apache...
5,google cloud platform,gcp,Back- End Developer,Confirmed: intended technical meaning,Reviewed contexts use GCP to refer to Google C...
6,kubernetes,k8s,Python Developer (Data-driven),Confirmed: intended technical meaning,Reviewed contexts use K8s to refer to Kubernet...
7,machine learning,ml,Data Scientist,Confirmed: intended technical meaning,Reviewed contexts use ML to refer to machine l...
8,mongodb,mongo,Junior Data Analyst,Confirmed: intended technical meaning,Reviewed contexts use Mongo to refer to MongoD...
9,natural language processing,nlp,Senior ML Engineer (GPT LNN NLP AI),Confirmed: intended technical meaning,Reviewed contexts use NLP to refer to natural ...


Completed alias context review for 13 candidate aliases.


## Final Candidate-Alias Decisions

Candidate aliases were assessed using development data only.

The final decision for alias selection considered not only frequency but  reviewed technical meaning, equivalence to the canonical concept, extraction value, redundancy, and compatibility with the frozen matching policy.

In [22]:
aliasDecisionRows = [
    (
        "amazon web services",
        "aws",
        "Accepted",
        "Reviewed development contexts refer to Amazon Web Services cloud services."
    ),
    (
        "google cloud platform",
        "gcp",
        "Accepted",
        "Reviewed development contexts refer to Google Cloud Platform."
    ),
    (
        "postgresql",
        "postgres",
        "Accepted",
        "Reviewed development contexts refer to PostgreSQL databases."
    ),
    (
        "mongodb",
        "mongo",
        "Accepted",
        "Reviewed development contexts refer to MongoDB databases."
    ),
    (
        "apache spark",
        "spark",
        "Accepted",
        "Reviewed development contexts refer to Apache Spark data-processing technology."
    ),
    (
        "apache spark",
        "pyspark",
        "Accepted",
        "Reviewed development contexts refer to the PySpark Apache Spark interface."
    ),
    (
        "apache kafka",
        "kafka",
        "Accepted",
        "Reviewed development contexts refer to Apache Kafka messaging or streaming technology."
    ),
    (
        "apache airflow",
        "airflow",
        "Accepted",
        "Reviewed development contexts refer to Apache Airflow workflow orchestration."
    ),
    (
        "kubernetes",
        "k8s",
        "Accepted",
        "Reviewed development contexts refer to Kubernetes container orchestration."
    ),
    (
        "power bi",
        "powerbi",
        "Accepted",
        "One reviewed development context clearly refers to the Power BI business-intelligence tool."
    ),
    (
        "scikit-learn",
        "sklearn",
        "Accepted",
        "One reviewed development context clearly refers to the scikit-learn machine-learning library."
    ),
    (
        "machine learning",
        "ml",
        "Accepted",
        "Reviewed development contexts use ML to refer to machine learning."
    ),
    (
        "natural language processing",
        "nlp",
        "Accepted",
        "Reviewed development contexts use NLP to refer to natural language processing."
    ),
    (
        "power bi",
        "power-bi",
        "Rejected",
        "No development-document matches were found."
    ),
    (
        "power bi",
        "pbi",
        "Rejected",
        "No development-document matches were found."
    ),
    (
        "power bi",
        "microsoft power bi",
        "Rejected as redundant",
        "The canonical phrase power bi is already matched within the longer expression."
    )
]


aliasDecisions = pd.DataFrame(
    aliasDecisionRows,
    columns=[
        "Canonical label",
        "Candidate alias",
        "Final decision",
        "Reviewed rationale"
    ]
)


coverageCounts = {
    row[
        "Candidate surface form"
    ]: row[
        "Development document count"
    ]
    for _, row in coverageAudit.iterrows()
}


powerBiCoverageCounts = {
    row[
        "Candidate form"
    ]: row[
        "Development document count"
    ]
    for _, row in powerBiAlternateAudit.iterrows()
}


aliasDecisions[
    "Development document count"
] = aliasDecisions[
    "Candidate alias"
].apply(
    lambda alias: (
        coverageCounts[
            alias
        ]
        if alias in coverageCounts
        else powerBiCoverageCounts.get(
            alias,
            pd.NA
        )
    )
)


assert aliasDecisions.loc[
    aliasDecisions[
        "Candidate alias"
    ].eq(
        "aws"
    ),
    "Development document count"
].iloc[
    0
] == 72, (
    "Expected aws to occur in 72 development documents."
)


assert aliasDecisions.loc[
    aliasDecisions[
        "Candidate alias"
    ].eq(
        "gcp"
    ),
    "Development document count"
].iloc[
    0
] == 15, (
    "Expected gcp to occur in 15 development documents."
)


assert aliasDecisions.loc[
    aliasDecisions[
        "Candidate alias"
    ].eq(
        "spark"
    ),
    "Development document count"
].iloc[
    0
] == 27, (
    "Expected spark to occur in 27 development documents."
)


assert aliasDecisions.loc[
    aliasDecisions[
        "Candidate alias"
    ].eq(
        "ml"
    ),
    "Development document count"
].iloc[
    0
] == 28, (
    "Expected ml to occur in 28 development documents."
)


acceptedAliasRows = aliasDecisions.loc[
    aliasDecisions[
        "Final decision"
    ].eq(
        "Accepted"
    )
].copy()


rejectedAliasRows = aliasDecisions.loc[
    aliasDecisions[
        "Final decision"
    ].eq(
        "Rejected"
    )
].copy()


redundantAliasRows = aliasDecisions.loc[
    aliasDecisions[
        "Final decision"
    ].eq(
        "Rejected as redundant"
    )
].copy()


assert len(aliasDecisions) == 16, (
    "Expected 16 candidate alias decisions."
)


assert len(acceptedAliasRows) == 13, (
    "Expected 13 accepted aliases."
)


assert len(rejectedAliasRows) == 2, (
    "Expected 2 rejected aliases."
)


assert len(redundantAliasRows) == 1, (
    "Expected 1 redundant-form decision."
)


aliasDecisionSummary = pd.DataFrame(
    {
        "Decision": [
            "Candidate alias decisions",
            "Accepted aliases",
            "Rejected aliases",
            "Rejected as redundant"
        ],
        "Count": [
            len(aliasDecisions),
            len(acceptedAliasRows),
            len(rejectedAliasRows),
            len(redundantAliasRows)
        ]
    }
)


display(
    aliasDecisionSummary
)


display(
    aliasDecisions.sort_values(
        [
            "Final decision",
            "Canonical label",
            "Candidate alias"
        ]
    ).reset_index(
        drop=True
    )
)


aliasDecisions.to_csv(
    runtimeFolder
    / "AliasDecisions.csv",
    index=False
)


print(
    "Accepted aliases: "
    f"{len(acceptedAliasRows)}"
)


print(
    "Rejected aliases: "
    f"{len(rejectedAliasRows)}"
)


print(
    "Rejected as redundant: "
    f"{len(redundantAliasRows)}"
)

,Decision,Count
0,Candidate alias decisions,16
1,Accepted aliases,13
2,Rejected aliases,2
3,Rejected as redundant,1


,Canonical label,Candidate alias,Final decision,Reviewed rationale,Development document count
0,amazon web services,aws,Accepted,Reviewed development contexts refer to Amazon ...,72
1,apache airflow,airflow,Accepted,Reviewed development contexts refer to Apache ...,16
2,apache kafka,kafka,Accepted,Reviewed development contexts refer to Apache ...,26
3,apache spark,pyspark,Accepted,Reviewed development contexts refer to the PyS...,7
4,apache spark,spark,Accepted,Reviewed development contexts refer to Apache ...,27
5,google cloud platform,gcp,Accepted,Reviewed development contexts refer to Google ...,15
6,kubernetes,k8s,Accepted,Reviewed development contexts refer to Kuberne...,3
7,machine learning,ml,Accepted,Reviewed development contexts use ML to refer ...,28
8,mongodb,mongo,Accepted,Reviewed development contexts refer to MongoDB...,4
9,natural language processing,nlp,Accepted,Reviewed development contexts use NLP to refer...,8


Accepted aliases: 13
Rejected aliases: 2
Rejected as redundant: 1


## Frozen Lexicon Specification

The final lexicon contains 20 canonical labels and 13 approved aliases.

System A searches only canonical forms. System B searches the same canonical
forms plus the approved aliases, while always returning canonical output labels.

The normalisation and boundary policy is frozen here. Ordinary hyphens are
preserved. A candidate directly attached to an ordinary hyphen is blocked by
the conservative boundary policy; for example, `ML-based` and `NLP-based` do
not match under this frozen policy.

No lexicon, alias, normalisation, or boundary-policy change is made after this
point.

In [23]:
acceptedAliasesByCanonicalLabel = {
    canonicalLabel: sorted(
        acceptedAliasRows.loc[
            acceptedAliasRows[
                "Canonical label"
            ].eq(
                canonicalLabel
            ),
            "Candidate alias"
        ].tolist()
    )
    for canonicalLabel in finalCanonicalInventory
}


generatedLexiconSpecification = {
    "schema version": "1.0",
    "status": "frozen for system implementation",
    "frozen using": (
        "Development subset only: 200 documents. "
        "Canonical and alias decisions are recorded in this notebook."
    ),
    "system a": {
        "description": (
            "Matches canonical forms only."
        )
    },
    "system b": {
        "description": (
            "Matches canonical forms and accepted aliases. "
            "Every match returns the canonical label."
        )
    },
    "normalisation policy": {
        "unicode normalisation": "NFKC",
        "lowercase": True,
        "apostrophe normalisation": (
            "Curly apostrophes are converted to straight apostrophes."
        ),
        "dash normalisation": (
            "En dashes and em dashes are converted to ASCII hyphens."
        ),
        "whitespace": (
            "Repeated whitespace is collapsed to one space."
        ),
        "ordinary hyphens": (
            "Preserved; ordinary hyphens are not converted to spaces."
        )
    },
    "boundary policy": {
        "left boundary": (
            r"(?<![A-Za-z0-9+#&_.\-])"
        ),
        "right boundary": (
            r"(?![A-Za-z0-9+#&_\-])"
        ),
        "internal spaces": (
            "One or more whitespace characters may represent a space "
            "inside a multiword form."
        ),
        "hyphen policy": (
            "A candidate directly joined to a hyphen is blocked. "
            "Therefore ML-based and NLP-based do not match "
            "under the conservative baseline policy."
        )
    },
    "skills": {
        canonicalLabel: {
            "aliases": acceptedAliasesByCanonicalLabel[
                canonicalLabel
            ]
        }
        for canonicalLabel in finalCanonicalInventory
    },
    "rejected or redundant forms": {
        "power-bi": (
            "Rejected because it had zero development-document matches."
        ),
        "pbi": (
            "Rejected because it had zero development-document matches."
        ),
        "microsoft power bi": (
            "Rejected as redundant because canonical power bi "
            "already appears in the longer expression."
        )
    }
}


generatedCanonicalLabels = set(
    generatedLexiconSpecification[
        "skills"
    ].keys()
)


generatedAliasCount = sum(
    len(
        details[
            "aliases"
        ]
    )
    for details in generatedLexiconSpecification[
        "skills"
    ].values()
)


assert generatedCanonicalLabels == set(
    finalCanonicalInventory
), (
    "The generated lexicon canonical labels do not match "
    "the final inventory."
)


assert len(generatedCanonicalLabels) == 20, (
    "The generated lexicon must contain 20 canonical labels."
)


assert generatedAliasCount == 13, (
    "The generated lexicon must contain 13 accepted aliases."
)


generatedLexiconPath = (
    runtimeFolder
    / "GeneratedLexiconSpecification.json"
)


with generatedLexiconPath.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        generatedLexiconSpecification,
        file,
        indent=2,
        ensure_ascii=False
    )


lexiconSummary = pd.DataFrame(
    {
        "Measure": [
            "Canonical labels",
            "Approved System B aliases",
            "Rejected aliases",
            "Rejected redundant forms"
        ],
        "Count": [
            len(generatedCanonicalLabels),
            generatedAliasCount,
            len(rejectedAliasRows),
            len(redundantAliasRows)
        ]
    }
)


display(
    lexiconSummary
)


print(
    "Generated lexicon specification created successfully."
)


print(
    f"Canonical labels: {len(generatedCanonicalLabels)}"
)


print(
    f"Approved aliases: {generatedAliasCount}"
)

,Measure,Count
0,Canonical labels,20
1,Approved System B aliases,13
2,Rejected aliases,2
3,Rejected redundant forms,1


Generated lexicon specification created successfully.
Canonical labels: 20
Approved aliases: 13


## Verification Against the Committed Frozen Lexicon

The preceding sections generated a lexicon specification from the documented
development-only canonical and alias decisions.

This final check verifies that the generated canonical-label and alias mapping
exactly matches the committed frozen lexicon used by Notebook 2 for testing and
by Notebook 4 for held-out evaluation.

The committed file is read here for verification only. It is not modified.

In [24]:
committedLexiconPath = (
    configFolder
    / "LexiconList20_final.json"
)


assert committedLexiconPath.exists(), (
    "The committed frozen lexicon file was not found."
)


with committedLexiconPath.open(
    "r",
    encoding="utf-8"
) as file:

    committedLexicon = json.load(
        file
    )


committedSkills = committedLexicon[
    "skills"
]


committedCanonicalLabels = set(
    committedSkills.keys()
)


committedAliasMapping = {
    canonicalLabel: sorted(
        details[
            "aliases"
        ]
    )
    for canonicalLabel, details in committedSkills.items()
}


generatedAliasMapping = {
    canonicalLabel: sorted(
        details[
            "aliases"
        ]
    )
    for canonicalLabel, details in generatedLexiconSpecification[
        "skills"
    ].items()
}


committedAliasCount = sum(
    len(
        aliases
    )
    for aliases in committedAliasMapping.values()
)


assert committedCanonicalLabels == generatedCanonicalLabels, (
    "The generated and committed canonical-label sets differ."
)


assert committedAliasMapping == generatedAliasMapping, (
    "The generated and committed canonical-to-alias mappings differ."
)


assert len(committedCanonicalLabels) == 20, (
    "The committed frozen lexicon must contain 20 canonical labels."
)


assert committedAliasCount == 13, (
    "The committed frozen lexicon must contain 13 aliases."
)


lexiconVerificationSummary = pd.DataFrame(
    {
        "Verification": [
            "Generated canonical-label count",
            "Committed canonical-label count",
            "Generated alias count",
            "Committed alias count",
            "Canonical-label mapping identical",
            "Canonical-to-alias mapping identical"
        ],
        "Result": [
            len(generatedCanonicalLabels),
            len(committedCanonicalLabels),
            generatedAliasCount,
            committedAliasCount,
            True,
            True
        ]
    }
)


display(
    lexiconVerificationSummary
)


print(
    "Generated lexicon specification exactly matches "
    "the committed frozen lexicon."
)

,Verification,Result
0,Generated canonical-label count,20
1,Committed canonical-label count,20
2,Generated alias count,13
3,Committed alias count,13
4,Canonical-label mapping identical,True
5,Canonical-to-alias mapping identical,True


Generated lexicon specification exactly matches the committed frozen lexicon.


## Conclusion

This notebook reproduced the development-side design of the experiment from a
pinned source-dataset revision.

It created a deterministic filtered corpus of 9,222 postings, separated 200
development documents from 100 employer-disjoint held-out evaluation documents,
and used only the development subset for candidate discovery, coverage auditing,
canonical-inventory selection, and alias decisions.

The final frozen lexicon contains:

- 20 canonical technical-skill labels.
- 13 approved System B aliases.
- 2 rejected candidate aliases.
- 1 rejected redundant candidate form.
- A conservative normalisation and boundary policy.

The generated canonical-label and alias mapping exactly matches the committed
frozen lexicon used in subsequent notebooks.

Notebook 2 implements and tests the frozen matcher. Notebook 3 documents the
independent gold-annotation workflow. Notebook 4 evaluates both systems on the
held-out 100-document corpus.